# Orbital rotation: `exp(-κ)` and parameter packing on top of `rotate_tensor`

`rotate_tensor(u, tensor)` changes the single-particle basis of any 2k-index tensor.
What an orbital-optimisation loop needs on top of that is the standard parameterisation
of `u`: an anti-Hermitian generator κ with `U = exp(-κ)`, and a flat real vector of
κ's free entries that an optimiser can drive.  Three helpers in
`qarp.operators` do exactly that and nothing more:

- `orbital_rotation_matrix(kappa)` — `U = expm(-κ)`; new orbitals are the columns of `U`,
  so `C_new = C_old @ U`.
- `orbital_rotation_generator(parameters, n)` — real skew-symmetric κ from its
  `n(n-1)/2` strict-lower-triangle entries (`np.tril_indices(n, -1)` order).
- `orbital_rotation_parameters(kappa)` — the inverse packing.

The optimiser and the energy functional are yours.  This notebook perturbs the RHF
orbitals of LiH, then recovers the RHF energy by minimising the closed-shell mean-field
energy over the packed parameters with `scipy.optimize.minimize`.

In [ ]:
import numpy as np
from pyscf import gto, scf
from scipy.optimize import minimize

from qarp.operators import rotate_tensor
from qarp.operators.pyscf import integrals_from_mf
from qarp.operators import (
    orbital_rotation_generator,
    orbital_rotation_matrix,
    orbital_rotation_parameters,
)

## 1. Reference: LiH / STO-3G at the RHF minimum

`integrals_from_mf` returns `(constant, h1, h2)` in the MO basis, chemists' notation.

In [ ]:
mol = gto.M(atom="Li 0 0 0; H 0 0 1.3", basis="sto3g", verbose=0)
mf = scf.RHF(mol).run()
constant, h1, h2 = integrals_from_mf(mf)
n = h1.shape[0]
n_occupied = mol.nelectron // 2
n_parameters = n * (n - 1) // 2
print(f"{n} orbitals, {n_occupied} doubly occupied, {n_parameters} rotation parameters")
print(f"E_RHF = {mf.e_tot:.10f} Ha")

## 2. The helpers

A 2×2 generator makes the convention concrete: `κ = [[0, -θ], [θ, 0]]` gives the
rotation by `-θ`, so the sign of the exponent is visible, not hidden.

In [ ]:
theta = 0.3
u_2x2 = orbital_rotation_matrix([[0.0, -theta], [theta, 0.0]])
print(np.round(u_2x2, 6))
print("cos θ =", round(np.cos(theta), 6), " sin θ =", round(np.sin(theta), 6))

# Packing: the parameter vector fills the strict lower triangle, row by row.
kappa = orbital_rotation_generator([0.1, 0.2, 0.3], 3)
print(kappa)
print(orbital_rotation_parameters(kappa))

For real skew-symmetric κ, `U` is real orthogonal with `det U = +1`, which is what
`rotate_tensor` needs to act on chemists'-notation integrals.

In [ ]:
rng = np.random.default_rng(1234)
u = orbital_rotation_matrix(orbital_rotation_generator(rng.normal(size=n_parameters), n))
print("U^T U = I:", np.allclose(u.T @ u, np.eye(n), atol=1e-12), " det U =", round(np.linalg.det(u), 12))

## 3. Perturb the orbitals, then recover the RHF energy

The closed-shell mean-field energy of a set of integrals, in chemists' notation, is

$$E = c + 2\sum_i h_{ii} + \sum_{ij}\bigl[2\,(ii|jj) - (ij|ji)\bigr], \qquad i, j \in \text{occupied}.$$

Starting from orbitals rotated by a random κ₀ with `‖κ₀‖_F = 0.1`, the minimum over the
packed parameters must come back to `mf.e_tot`.  Occupied–occupied and virtual–virtual
rotations leave `E` invariant, so those directions are flat; BFGS simply never moves
along them.

In [ ]:
def closed_shell_energy(constant, h1, h2, n_occupied):
    occ = range(n_occupied)
    one = 2 * sum(h1[i, i] for i in occ)
    two = sum(2 * h2[i, i, j, j] - h2[i, j, j, i] for i in occ for j in occ)
    return constant + one + two


x0 = rng.standard_normal(n_parameters)
x0 *= 0.1 / np.linalg.norm(orbital_rotation_generator(x0, n))
u0 = orbital_rotation_matrix(orbital_rotation_generator(x0, n))
h1_perturbed, h2_perturbed = rotate_tensor(u0, h1), rotate_tensor(u0, h2)


def energy(x):
    u = orbital_rotation_matrix(orbital_rotation_generator(x, n))
    return closed_shell_energy(constant, rotate_tensor(u, h1_perturbed), rotate_tensor(u, h2_perturbed), n_occupied)


print(f"E at the perturbed start : {energy(np.zeros(n_parameters)):.10f} Ha")
result = minimize(energy, np.zeros(n_parameters), method="BFGS", options={"gtol": 1e-6})
print(f"E after optimisation     : {result.fun:.10f} Ha   ({result.nfev} evaluations)")
print(f"E_RHF                    : {mf.e_tot:.10f} Ha")
print(f"|ΔE| = {abs(result.fun - mf.e_tot):.2e} Ha")

## 4. The optimised orbitals

The new MO coefficients are `C_old @ U` — the same side and sign that
`test_rotation_invariance` pins against pyscf's CASSCF.  Re-extracting the integrals
from pyscf with those coefficients gives the same tensors as `rotate_tensor` did.

In [ ]:
u_opt = orbital_rotation_matrix(orbital_rotation_generator(result.x, n))
mf.mo_coeff = mf.mo_coeff @ u0 @ u_opt
_, h1_pyscf, h2_pyscf = integrals_from_mf(mf)
print("h1 agrees:", np.linalg.norm(h1_pyscf - rotate_tensor(u_opt, h1_perturbed)) < 1e-10)
print("h2 agrees:", np.linalg.norm(h2_pyscf - rotate_tensor(u_opt, h2_perturbed)) < 1e-10)

## 5. Composing with the circuit side

`OrbitalRotationBlock` emits the Jordan–Wigner Givens circuit of a real orthogonal mode
rotation, over *spin orbitals* in abab order.  A spatial `U` is expanded first with
`spatial_to_spin_orbital`, whose k = 1 case is the block-diagonal abab matrix the block
expects — so the same κ that rotates the integrals classically also drives the circuit.

In [ ]:
from qarp.blocks import OrbitalRotationBlock
from qarp.operators.integrals import spatial_to_spin_orbital

u_spin = spatial_to_spin_orbital(u_opt)
block = OrbitalRotationBlock(u_spin)
print("spin-orbital rotation:", u_spin.shape, " block on", block.n_qubits, "qubits")